# **Simulation Hedy 2025**

Team: TU Wien Space Team \
Project: Lamarr \
Rocket: Hedy


## Installs

In this section all needed libraries are installed and the needed classes imported

In [ ]:
# %pip install rocketpy==1.12.1
# %pip install CoolProp
# %pip install openmeteo-requests requests-cache retry-requests pandas plotly colorama
# %pip install --upgrade nbformat
# %pip install "niquests==3.18.8" "urllib3-future==2.20.904"

In [ ]:
from rocketpy import (
    Environment,
    Rocket,
    Flight,
    Fluid, 
    SolidMotor,
    LiquidMotor,
    CylindricalTank,
    MassFlowRateBasedTank,
    NoseCone,
    RailButtons,
    Tail,
    Parachute,
    TrapezoidalFins,
    FreeFormFins,
    Function
)
from rocketpy.simulation import FlightDataExporter
import CoolProp.CoolProp as CP

import numpy as np
import pandas as pd
from math import pi
import datetime
from pathlib import Path
from colorama import Fore, Style                                  # https://github.com/tartley/colorama


import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))   # walk up 1 level to root folder
from simulation.export_weather_data import export_weather_data
from simulation.custom_print_and_plot_functions import CustomPlots, CustomPrints

In [ ]:
# reload imported Python modules when you change their .py files
%load_ext autoreload
%autoreload 2

## Configuration

The length unit chosen here is millimeters to keep the values more readable. If necessary, values should be converted accordingly.

In [ ]:
use_standard_env = False

# Rocket configuration data
env_config = {
    "date" : (2025, 10, 12, 17, 00, 00),  # yyyy, mm, dd, hh, mm, ss    # preset
    "latitude" : 39.232465,                # Launch latitude             # preset
    "longitude" : -8.172108,               # Launch longitude            # preset
    "timezone" : "Europe/Lisbon"          # GMT+1                       # preset
}
motor_config = {
    "holddown_time" : 1.1,                # s       # preset

    "nitrogen_tank" : {
        "length" : 235,                   # mm      # measured
        "CG_lox" : 2653,                  # mm      # measured # CG lox nitrogen tank
        "CG_ethanol" : 1450,              # mm      # measured # CG ethanol nitrogen tank
        "outer_diameter": 93,             # mm      # measured
        "volume" : 1.2,                   # l       # measured
        "massflow" : 0.03,                # kg/s    # estimate
    },
    "ethanol_tank" : {
        "length" : 573,                   # mm      # measured
        "CG" : 852,                       # mm      # measured
        "outer_diameter": 115,            # mm      # measured
        "volume" : 5.27,                  # l       # measured
        "massflow" : 0.45,                # kg/s    # tested
    },
    "lox_tank" : {
        "length" : 573,                   # mm      # measured
        "CG" : 1960,                      # mm      # measured
        "outer_diameter" : 115,           # mm      # measured
        "volume" : 5.27,                  # l       # measured
        "massflow" : 0.56,                # kg/s    # tested
    },
    "nozzle" : {
        "diameter" : 75,                  # mm      # measured
        "position" : -40,                 # mm      # measured
    },

    # "thrust": "thrust_euroc_launch.csv",               # N       # measured
    "thrust": "thrust.csv",
    # "thrust": "thrust_08_25.csv",         # N       # measured (static fire test)

    "temperature" : {
        "ethanol" : 298.15,               # K       # preset (= 25°C)
        "lox" : 93.15,                    # K       # preset (= -180°C)
        "nitrogen" : 298.15               # K       # preset (= 25°C)
    },
    "pressure" : {
        "ethanol" : 3000000,              # Pa     # preset
        "lox" : 3000000,                  # Pa     # preset
        "nitrogen" : 30000000             # Pa     # preset
    }
}

rocket_config = {
    "total_weight" : 16200,               # g       # weighed dry mass
    "total_length" : 3707,                # mm      # measured
    "total_CG" : 1813,                    # mm      # measured
    "moment_of_intertia_Z" : 0.0255,      # kg*m^2  # calculated
    "moment_of_intertia_XY" : 16.8,       # kg*m^2  # calculated
    "nosecone" : {
        "length" : 561,                   # mm      # measured    # 575 + 66 - 80 = 561
        "kind" : "lv haack"
    },
    "railbuttons" : {
        "upper" : 1646,                   # mm      # measured
        "lower" : 305,                    # mm      # measured
    },
    "tailcone" : {
        "diameter" : 108,                 # mm      # measured
        "length" : 245,                   # mm      # measured
        "cylindrical_height" : 35,        # mm      # measured
    },
    "rocket" : {
        "thickness" : 1.4,                # mm      # measured
        "diameter" : 132.8,               # mm      # measured
    },
    "fins" : {
        "name" : "Biconvex",
        "amount" : 4,
        "root_chord" : 250,               # mm      # measured
        "tip_chord" : 55,                 # mm      # measured
        "span" : 110,                     # mm      # measured
        "sweep_length" : 193,             # mm      # measured
        "position" : 250,                 # mm      # measured
        "shape_points" : ((0,0),
                          (0.250,-0.012),
                          (0.250, 0.108),
                          (0.195, 0.108),
                          (0,0))          # m     # measured
    },
    "parachutes" : {
        "main" : {
            "cd_s" : 6.911503837897546,                     # calculated
            # "cd": 1.8,
            # "radius": 2.1 / 2,                                      # m                 of projected area
            "trigger" : 450,              # m               # preset
            "sampling_rate" : 105,        # hz              # preset
            "lag" : 4,                    # s               # measured
            "noise" : (0, 8.3, 0.5)       # (pa, pa, pa)    # preset
        },
        "drogue" : {
            "cd_s" : 0.336875,
            # "radius": 0.4415481,                                   # m     for a circle with area A_fabric (we use fabric since the projected area is almost the one from the fabric)
            # "fabric_area": 0.35**2 * 5,                            # m²
            "trigger" : "apogee",         # m               # preset
            "sampling_rate" : 105,        # hz              # preset
            "lag" : 1,                    # s               # measured
            "noise" : (0, 8.3, 0.5)       # (pa, pa, pa)    # preset
        }
    }
}
flight_config = {
    "rail_length" : 11,                   # m       #Preset
    "inclination" : 84,                   # °       #Preset
    "heading" : 184,                      # °       #Preset
    "terminate_on_apogee" : False,
}

## Environments Initialization




In this Section the environments are initialized.
*   **envForecast**: environment with the weather data from the wyoming-sounding data source at the location and date of EuRoc
*   **envNormal**: normalized environment with standard atmospheric values at the time and location of EuRoc
*   **envCustom**: custom environment with variable values for temperature and windspeed


In [ ]:
#Ponte de Sor: 39.12368, -8.03333
#EUROC: 09.-15.10.2025
#possible launch date: 11.10.2025


def print_ev(env):
    # env.prints.gravity_details()
    env.prints.launch_site_details()
    env.prints.atmospheric_model_details()
    env.prints.atmospheric_conditions()
    # env.prints.print_earth_details()
    
    
latitude   = env_config["latitude"]
longitude  = env_config["longitude"]
timezone   = env_config["timezone"]
date       = env_config["date"]
environments = {}

if use_standard_env:
    # --- Standard Atmosphere Environment ---
    env_Standardized = Environment()
    env_Standardized.set_location(latitude=latitude, longitude=longitude)
    env_Standardized.set_elevation("Open-Elevation")         # API currently not working
    # env_Standardized.set_elevation(altitude)
    env_Standardized.set_date(date, timezone=timezone)
    env_Standardized.set_atmospheric_model(type="standard_atmosphere")
    
    print("Standard Atmosphere Environment:")
    print_ev(env_Standardized)
    env_Standardized.plots.atmospheric_model()
    environments["Standardized"] = env_Standardized

else:
    # --- Reanalysis Environment ---
    # Environment based on Forecast data for the EuRoC 2025
    env_Reanalysis = Environment()

    env_Reanalysis.set_location(latitude = latitude, longitude = longitude)
    env_Reanalysis.set_elevation("Open-Elevation")
    env_Reanalysis.set_date(date, timezone = timezone)

    env_Reanalysis.set_atmospheric_model(
        type="Reanalysis",
        file="euroc_weather.nc",
        dictionary="ECMWF",
    )
    print("Reanalysis Environment:")
    print_ev(env_Reanalysis)
    env_Reanalysis.plots.atmospheric_model()
    environments["Reanalysis"] = env_Reanalysis


## Simulation

### Tanks / Engine



In [ ]:
#holddown time
t_holddown          = motor_config["holddown_time"]                                     # s

# tank height
h_nitrogen_tank     = motor_config["nitrogen_tank"]["length"]     / 1000                # m
h_ethanol_tank      = motor_config["ethanol_tank"]["length"]      / 1000                # m
h_lox_tank          = motor_config["lox_tank"]["length"]          / 1000                # m
# OuterDiameter
OD_nitrogen_tank    = motor_config["nitrogen_tank"]["outer_diameter"]   / 1000          # m
OD_ethanol_tank     = motor_config["ethanol_tank"]["outer_diameter"]    / 1000          # m
OD_lox_tank         = motor_config["lox_tank"]["outer_diameter"]        / 1000          # m
# volume
v_nitrogen_tank     = motor_config["nitrogen_tank"]["volume"]   / 1000                  # m
v_ethanol_tank      = motor_config["ethanol_tank"]["volume"]    / 1000                  # m
v_lox_tank          = motor_config["lox_tank"]["volume"]        / 1000                  # m
# massflows
mdot_nitrogen       = motor_config["nitrogen_tank"] ["massflow"]                        # kg/s
mdot_ethanol        = motor_config["ethanol_tank"]["massflow"]                          # kg/s
mdot_lox            = motor_config["lox_tank"]["massflow"]                              # kg/s
# nozzle
nozzle_diameter     = motor_config["nozzle"]["diameter"]          / 1000                # m

#thrust
thrust              = motor_config["thrust"]                                            # N

# Temperature Lox & Ethanol
T_nitrogen          = motor_config["temperature"]["nitrogen"]                           # K  (=25°C)
T_ethanol           = motor_config["temperature"]["ethanol"]                            # K  (=25°C)
T_lox               = motor_config["temperature"]["lox"]                                # K
# pressure LOX & Ethanol
p_nitrogen          = motor_config["pressure"]["nitrogen"]                              # Pa
p_ethanol           = motor_config["pressure"]["ethanol"]                               # Pa
p_lox               = motor_config["pressure"]["lox"]                                   # Pa


# Propellants

# define density
rho_nitrogen = CP.PropsSI("D","T",T_nitrogen,"P|gas",p_nitrogen,"N2")                   # kg/m^3
rho_ethanol = CP.PropsSI("D", "T|liquid", T_ethanol, "P", p_ethanol, "ethanol")         # kg/m^3
rho_lox = CP.PropsSI("D", "T|liquid", T_lox, "P", p_lox, "oxygen")                      # kg/m^3

# define fluids
nitrogen = Fluid(name = "N2", density = rho_nitrogen)
ethanol = Fluid(name = "ethanol", density = rho_ethanol)
lox = Fluid(name = "LOX", density = rho_lox)

# define tanks geometry
nitrogen_tank_shape = CylindricalTank(radius = OD_nitrogen_tank / 2, height = h_nitrogen_tank, spherical_caps = True)
ethanol_tank_shape = CylindricalTank(radius = OD_ethanol_tank / 2, height = h_ethanol_tank, spherical_caps = True)
lox_tank_shape = CylindricalTank(radius = OD_lox_tank / 2, height = h_lox_tank, spherical_caps = True)


m_nitrogen = v_nitrogen_tank * rho_nitrogen                   # m
m_ethanol = v_ethanol_tank * rho_ethanol                      # m
m_lox = v_lox_tank * rho_lox                                  # m

t_burn_nitrogen = (m_nitrogen / mdot_nitrogen)-0.001          # s


t_burn_ethanol = (m_ethanol / mdot_ethanol)
t_burn_lox = (m_lox / mdot_lox)

# BOOKMARK: why are we doing this? why does it differ from Hedy_clippedDelta_FINAL?

if(t_burn_ethanol<t_burn_lox):
  t_burn = t_burn_ethanol
  print(f"using ethanol burn time ({t_burn_ethanol}). LOX burntime is {t_burn_lox}")
else:
  t_burn = t_burn_lox
  print(f"using lox burn time ({t_burn_lox}). Ethanol burntime is {t_burn_ethanol}")


# finally the burn time is updated to account for the holddown time; the mass which contributes to the flight is updated as well
t_burn -= t_holddown

m_lox = mdot_lox * t_burn + 0.0001                  # add 0.0001s to not get zero mass in tanks (rocketpy otherwise throws an error)
m_ethanol = mdot_ethanol * t_burn + 0.0001          # mdot = massflow
m_nitrogen = mdot_nitrogen * t_burn + 0.0001


# define tanks
nitrogen_tank = MassFlowRateBasedTank(
    name = "nitrogen tank",
    geometry = nitrogen_tank_shape,
    flux_time = t_burn,                                 # s
    initial_liquid_mass = 0,                            # kg
    initial_gas_mass = m_nitrogen,                      # kg
    liquid_mass_flow_rate_in = 0,                       # kg/s
    liquid_mass_flow_rate_out = 0,                      # kg/s
    gas_mass_flow_rate_in = 0,                          # kg/s
    gas_mass_flow_rate_out = lambda t: mdot_nitrogen,   # ks/s  why with lambda function? why not just = mdot_nitrogen? (some reason with constant nitrogen massflow)
    liquid = Fluid(name = "liquid", density = 0.0001),  # ignore
    gas = nitrogen,
)
ethanol_tank = MassFlowRateBasedTank(
    name = "fuel tank",
    geometry = ethanol_tank_shape,
    flux_time = t_burn,                                 # s
    initial_liquid_mass = m_ethanol,                    # kg
    initial_gas_mass = 0,                               # kg          # can be neglected
    liquid_mass_flow_rate_in = 0,                       # kg/s
    liquid_mass_flow_rate_out = lambda t: mdot_ethanol, # kg/s
    gas_mass_flow_rate_in = lambda t: mdot_nitrogen,    # kg/s
    gas_mass_flow_rate_out = 0,                         # kg/s
    liquid = ethanol,
    gas = nitrogen,
)

lox_tank = MassFlowRateBasedTank(
    name = "oxidizer tank",
    geometry = lox_tank_shape,
    flux_time = t_burn,                                 # s
    initial_liquid_mass = m_lox,                        # kg
    initial_gas_mass = 0,                               # kg          # can be neglected
    liquid_mass_flow_rate_in = 0,                       # kg/s
    liquid_mass_flow_rate_out = lambda t: mdot_lox,     # kg/s
    gas_mass_flow_rate_in = lambda t: mdot_nitrogen,    # kg/s
    gas_mass_flow_rate_out = 0,                         # kg/s
    liquid = lox,
    gas = nitrogen,
)

skuld = LiquidMotor(
    dry_mass = 0,                       # kg
    dry_inertia = (0,0,0),              # kg*m^2
    center_of_dry_mass_position = 0,    # m

    nozzle_radius = nozzle_diameter /2, # m
    nozzle_position = 0,                # m

    thrust_source = thrust,             # N
    burn_time = t_burn,                 # s
    coordinate_system_orientation = "nozzle_to_combustion_chamber"
)

skuld.add_tank(tank = ethanol_tank, position    = motor_config["ethanol_tank"]["CG"]            / 1000)
skuld.add_tank(tank = lox_tank, position        = motor_config["lox_tank"]["CG"]                / 1000)
skuld.add_tank(tank = nitrogen_tank, position   = motor_config["nitrogen_tank"]["CG_ethanol"]   / 1000)
skuld.add_tank(tank = nitrogen_tank, position   = motor_config["nitrogen_tank"]["CG_lox"]       / 1000)

# --- prints ---
skuld.prints.nozzle_details()
skuld.prints.motor_details()

# --- plots ---
skuld.draw()
skuld.plots.thrust()
# skuld.plots.mass_flow_rate()
# skuld.plots.exhaust_velocity()
skuld.plots.total_mass()
# skuld.plots.propellant_mass()
skuld.plots.center_of_mass()
# skuld.plots.burn_rate()
# skuld.plots.burn_area()
# skuld.plots.Kn()
skuld.plots.inertia_tensor()


### Rocket components

In [ ]:
#rocket
rocket_length     = rocket_config["total_length"]         / 1000
rocket_diameter   = rocket_config["rocket"]["diameter"]   / 1000
rocket_thickness  = rocket_config["rocket"]["thickness"]  / 1000


nosecone_length = (rocket_config["nosecone"]["length"]-80)    / 1000   # -80 mm to adjust for cylindrical section
nosecone_kind = rocket_config["nosecone"]["kind"]

nose_cone = NoseCone(
    length = nosecone_length,
    base_radius = rocket_diameter/2,
    kind = nosecone_kind
)

tailcone_cylindrical    = rocket_config["tailcone"]["cylindrical_height"] / 1000
tailcone_length         = rocket_config["tailcone"]["length"]             / 1000
tailcone_bottom_radius  = rocket_config["tailcone"]["diameter"]        /2 / 1000

tail = Tail(
    top_radius = rocket_diameter /2,
    bottom_radius = tailcone_bottom_radius,
    length = tailcone_length,
    rocket_radius = rocket_diameter/2
)

fin_amount        = rocket_config["fins"]["amount"]
fin_name          = rocket_config["fins"]["name"]
fin_position      = rocket_config["fins"]["position"]     / 1000
fin_root_chord    = rocket_config["fins"]["root_chord"]   / 1000
fin_tip_chord     = rocket_config["fins"]["tip_chord"]    / 1000
fin_span          = rocket_config["fins"]["span"]         / 1000
fin_sweep_length  = rocket_config["fins"]["sweep_length"] / 1000
fin_shape_points  = rocket_config["fins"]["shape_points"]

trapezoidal_fin_set = TrapezoidalFins(
    n = fin_amount,
    root_chord = fin_root_chord,      # m
    tip_chord = fin_tip_chord,        # m
    span = fin_span,                  # m
    sweep_length = fin_sweep_length,  # m
    name = fin_name,
    rocket_radius = rocket_diameter/2 # m
  )
trapezoidal_fin_set.draw()

fin_set = FreeFormFins(
   n = fin_amount,
   shape_points = fin_shape_points,         # m
   rocket_radius=tailcone_bottom_radius,    # m
   name = "Freeform"
)
fin_set.draw()

# fin_set.prints.identity()
fin_set.prints.geometry()
# fin_set.prints.lift()
# fin_set.plots.airfoil()
# fin_set.plots.roll()
# fin_set.plots.lift()


# fin_set = TrapezoidalFins(
#       n             = fin_amount,
#       root_chord    = 0.250,
#       tip_chord     = 0.050,
#       span          = 0.108,
#       sweep_length  = 0.200,
#       rocket_radius = tailcone_bottom_radius,
#       name          = "Trapezoidal"
# )
# fin_set.draw()


parachutes = {}

parachutes[0] = Parachute(
    name = "main",
    # cd_s = rocket_config["parachutes"]["drogue"]["cd_s"],                 # drag coeff per area (important)
    cd_s = rocket_config["parachutes"]["main"]["cd_s"],
    trigger = rocket_config["parachutes"]["main"]["trigger"],             # m (when parachute should be deployed)
    sampling_rate = rocket_config["parachutes"]["main"]["sampling_rate"], # hz (how often does the flight computer measure)
    lag = rocket_config["parachutes"]["main"]["lag"],                     # s (how long the parachute takes to open)
    noise = rocket_config["parachutes"]["main"]["noise"],                 # (pa, pa, pa)
)

parachutes[1] = Parachute(
    name = "drogue",
    cd_s = rocket_config["parachutes"]["drogue"]["cd_s"],
    trigger = rocket_config["parachutes"]["drogue"]["trigger"],             # m
    sampling_rate = rocket_config["parachutes"]["drogue"]["sampling_rate"], # hz
    lag = rocket_config["parachutes"]["drogue"]["lag"],                     # s
    noise = rocket_config["parachutes"]["drogue"]["noise"],                 # (pa, pa, pa)
)

for parachute in parachutes.values():
    CustomPlots.plot_parachute_model(parachute)

### Hedy
RocketPy definitions:
- dry mass = rocket with motor but without propellant
- Rocket Loaded Mass = Wet mass
- Rocket Center of Dry Mass - Nozzle Exit = Rocket Center of Dry Mass from bottom

In [ ]:

total_mass        = rocket_config["total_weight"] / 1000                    # kg

#inertia
inertia_x_y   = rocket_config["moment_of_intertia_XY"]                      # kg/m²
inertia_z     = rocket_config["moment_of_intertia_Z"]                       # kg/m²

upper_railbutton_position = rocket_config["railbuttons"]["upper"]/1000      # m
lower_railbutton_position = rocket_config["railbuttons"]["lower"]/1000      # m
nozzle_position     = motor_config["nozzle"]["position"]         / 1000     # m

#CG
CG = rocket_config["total_CG"] / 1000                                       # m

hedy = Rocket(
    radius = rocket_diameter /2,                        # m
    mass = total_mass,                                  # m
    inertia = (inertia_x_y, inertia_x_y, inertia_z),    # kg * m^2
    power_off_drag = "./power_off_drag.csv",            # from openrocket or CFD (first value mach number, second value drag coeff)
    power_on_drag = "./power_on_drag.csv",              # same as above
    center_of_mass_without_motor = CG,                  # m
    coordinate_system_orientation = "tail_to_nose"
)


hedy.add_motor(skuld, position = nozzle_position)


hedy.set_rail_buttons(upper_button_position= upper_railbutton_position, lower_button_position=lower_railbutton_position)

hedy.add_surfaces(surfaces=[nose_cone, fin_set, tail], positions=[rocket_length, fin_position, tailcone_length])

hedy.parachutes = list(parachutes.values())

hedy.prints.inertia_details()
# hedy.prints.rocket_geometrical_parameters()
# hedy.prints.rocket_aerodynamics_quantities()
hedy.prints.parachute_data()

# --- plots ---
hedy.plots.draw()
hedy.plots.total_mass()
# hedy.plots.reduced_mass()
hedy.plots.drag_curves()
# hedy.plots.static_margin()
# hedy.plots.stability_margin()
hedy.plots.thrust_to_weight()

### Flight

In [ ]:
rail_length           = flight_config["rail_length"]
inclination           = flight_config["inclination"]
heading               = flight_config["heading"]
terminate_on_apogee   = flight_config["terminate_on_apogee"]
flight_forecasts = {}

output_folder = Path("./trajectory_kml")
output_folder.mkdir(exist_ok=True)


for env_name, env in environments.items():
    print(Fore.GREEN + f"\n\n--- Simulating Flight in Environment: {env_name} ---" + Style.RESET_ALL)
    flight_forecast = Flight(
        rocket=hedy,
        environment=env,
        rail_length=rail_length,            # m
        inclination=inclination,            # angle
        heading=heading,                    # direction of the rocket (north, east, ...)
        terminate_on_apogee=terminate_on_apogee,
        name=env_name,
    )
    
    # --- prints ---
    # flight_forecast.prints.all()
    custom_prints = CustomPrints(flight_forecast)    
    flight_forecast.prints.launch_rail_conditions()
    flight_forecast.prints.out_of_rail_conditions()
    custom_prints.apogee_conditions()
    # flight_forecast.prints.apogee_conditions()
    custom_prints.parachute_events()
    # flight_forecast.prints.events_registered()
    flight_forecast.prints.impact_conditions()
    custom_prints.impact_coordinates()
    # flight_forecast.prints.maximum_values()

    
    # --- plots --- 
    # flight_forecast.plots.all()
    custom_plots = CustomPlots(
        flight_forecast=flight_forecast,
        motor=skuld,
        environment_name=env_name,
        rocket=hedy,
        rocket_config=rocket_config,
    )
    
    custom_plots.plot_stability_and_cg_cp_position()
    # flight_forecast.plots.stability_and_control_data()
    custom_plots.plot_angle_of_attack()
    custom_plots.plot_vertical_motion()
    flight_forecast.plots.trajectory_3d()
    
    
    file_name = output_folder / f"HEDY_Flight_Forecast_{env_name}.kml"
    FlightDataExporter(flight_forecast).export_kml(file_name=file_name, altitude_mode="relativetoground")
    flight_forecasts[env_name] = flight_forecast


In [ ]:
flight_forecast.export_data("flight_forecast_results.csv","x","y","z","e0","e1","e2","e3","w1","w2","w3", "latitude", "longitude", "altitude", "wind_velocity_x", "wind_velocity_y", "speed", "acceleration", "reynolds_number", "aerodynamic_lift", "aerodynamic_drag", "aerodynamic_bending_moment", "aerodynamic_spin_moment", "thrust_power", "drag_power", "stability_margin")

# **Stochastic Calculations (Monte Carlo)**


## Imports

In [ ]:
from rocketpy.stochastic import (
    StochasticEnvironment,
    StochasticSolidMotor,
    StochasticRocket,
    StochasticFlight,
    StochasticNoseCone,
    StochasticTail,
    StochasticTrapezoidalFins,
    StochasticParachute,
    StochasticRailButtons,
)
from rocketpy import MonteCarlo
from rocketpy import Components

## Stochastic environment

In [ ]:
envEnsemble = Environment()
envEnsemble.set_location(latitude = env_config["latitude"], longitude = env_config["longitude"])
envEnsemble.set_elevation("Open-Elevation")
envEnsemble.set_date((env_config["date"]), timezone= env_config["timezone"]) #Latest available date at time of simualtion
# envEnsemble.set_atmospheric_model(type = "Ensemble", file = "GEFS")

# EuRoC weather from 11.10.2025
envEnsemble.set_atmospheric_model(
    type="Reanalysis",
    file="euroc_weather.nc",
    dictionary="ECMWF",
)

stochastic_env = StochasticEnvironment(
    environment = envEnsemble,
    wind_velocity_x_factor = (0, 10, "normal"),
    wind_velocity_y_factor = (0, 10, "normal"),

    ensemble_member = list(range(envEnsemble.num_ensemble_members)),
)
stochastic_env.visualize_attributes()

## Approximating the liquid motor

In [ ]:
grain_outer_radius = 0.05  # m
grain_inner_radius = 0.01  # m
grain_height = 0.1         # m

approx_motor = SolidMotor(
    thrust_source = skuld.thrust_source,
    burn_time = skuld.burn_time,          # s
    center_of_dry_mass_position = 0,      # m
    dry_mass = skuld.dry_mass,            # kg
    dry_inertia = [0, 0, 0],              # kg * m^2
    grains_center_of_mass_position = CG,  # m
    grain_number = 1,
    grain_density = (lox_tank.fluid_mass(0) + ethanol_tank.fluid_mass(0)) / (((grain_outer_radius**2) * pi * grain_height) - ((grain_inner_radius**2) * pi * grain_height)), # kg / m^3
    grain_outer_radius = grain_outer_radius,         # m
    grain_initial_inner_radius = grain_inner_radius, # m
    grain_initial_height = grain_height,             # m
    grain_separation = 0,                     # m
    nozzle_radius = skuld.nozzle_radius,      # m
    nozzle_position = skuld.nozzle_position,  # m
    throat_radius = skuld.nozzle_radius,      # m
    reshape_thrust_curve = False,
    coordinate_system_orientation = 'nozzle_to_combustion_chamber',
)
#approx_motor.all_info()

## Stochastic rocket

In [ ]:
railbuttons = RailButtons(
    buttons_distance = upper_railbutton_position-lower_railbutton_position
)

In [ ]:
hedy.aerodynamic_surfaces = Components()
hedy.add_surfaces(surfaces=[nose_cone, trapezoidal_fin_set, tail], positions=[rocket_length, fin_position, tailcone_length])

stochastic_rocket = StochasticRocket(
    rocket = hedy,
)

stochastic_nose_cone = StochasticNoseCone(
    nosecone = nose_cone
)

stochastic_fin_set = StochasticTrapezoidalFins(
    trapezoidal_fins = trapezoidal_fin_set,
)

stochastic_tail = StochasticTail(
    tail = tail
)

stochastic_rail_buttons = StochasticRailButtons(
    rail_buttons = railbuttons
)

stochastic_main = StochasticParachute(
    parachute = parachutes[0]
)

stochastic_drogue = StochasticParachute(
    parachute = parachutes[1]
)

stochastic_motor = StochasticSolidMotor(
    solid_motor = approx_motor
)

stochastic_rocket.add_nose(stochastic_nose_cone)
stochastic_rocket.add_trapezoidal_fins(stochastic_fin_set)
stochastic_rocket.add_tail(stochastic_tail)
stochastic_rocket.add_parachute(stochastic_main)
stochastic_rocket.add_parachute(stochastic_drogue)
stochastic_rocket.add_motor(stochastic_motor)

stochastic_rocket.visualize_attributes()


stochastic_flight = StochasticFlight(
    flight = flight_forecast,
    inclination=(flight_config["inclination"], 1),
    heading=(flight_config["heading"], 2)
)

stochastic_flight.visualize_attributes()



test_dispersion = MonteCarlo(
    filename = "monte_carlo_class_example",
    environment = stochastic_env,
    rocket = stochastic_rocket,
    flight = stochastic_flight
)

## Simulation

In [ ]:
test_dispersion.simulate(number_of_simulations = 100, append = False)

print()
print()
print()
print()
print()
print()
print()
print()
print("//////////////////////////////////////////////////")
print("//                 FINISHED                     //")
print("//////////////////////////////////////////////////")

## Outputs

In [ ]:
print(test_dispersion.num_of_loaded_sims)
test_dispersion.prints.all()
test_dispersion.plots.ellipses(xlim = (-2000, 6000), ylim = (-7000, 3000))